In [ ]:
from __future__ import annotations
import os
import sys
import tomllib
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

In [ ]:
def find_repo_root(start: Path) -> Path:
    override = os.environ.get("HM_REPO_ROOT")
    if override:
        return Path(override).expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.toml").exists() or (candidate / "config.example.toml").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd().resolve())
CONFIG_PATH = REPO_ROOT / "config.toml"
if not CONFIG_PATH.exists():
    CONFIG_PATH = REPO_ROOT / "config.example.toml"

with CONFIG_PATH.open("rb") as fh:
    CONFIG = tomllib.load(fh)

DATASET_ROOT = (REPO_ROOT / CONFIG["pipeline"]["dataset_root"]).resolve()
TRANSACTIONS_PATH = DATASET_ROOT / CONFIG["sources"]["transactions"]
CUSTOMERS_PATH = DATASET_ROOT / CONFIG["sources"]["customers"]
ARTICLES_PATH = DATASET_ROOT / CONFIG["sources"]["articles"]

SPARK_CFG = CONFIG.get("spark", {})
PROCESSED_ROOT = (REPO_ROOT / SPARK_CFG.get("processed_root", "s3/processed")).resolve()
ANALYTICS_ROOT = (REPO_ROOT / SPARK_CFG.get("analytics_root", "s3/analytics")).resolve()
ROW_LIMIT = int(os.environ.get("HM_ROW_LIMIT", SPARK_CFG.get("row_limit", 0)) or 0)

for label, path in {
    "transactions": TRANSACTIONS_PATH,
    "customers": CUSTOMERS_PATH,
    "articles": ARTICLES_PATH,
}.items():
    if not path.is_file():
        raise FileNotFoundError(f"{label} file not found: {path}")

print("repo root :", REPO_ROOT)
print("config    :", CONFIG_PATH)
print("dataset   :", DATASET_ROOT)
print("row limit :", ROW_LIMIT or "all rows")
print("processed :", PROCESSED_ROOT)
print("analytics :", ANALYTICS_ROOT)


In [ ]:
if os.name == "nt":
    hadoop_home = Path(os.environ.get("HADOOP_HOME", r"C:\hadoop"))
    if (hadoop_home / "bin" / "winutils.exe").exists():
        os.environ["HADOOP_HOME"] = str(hadoop_home)
        os.environ["PATH"] = str(hadoop_home / "bin") + os.pathsep + os.environ["PATH"]
        print("HADOOP_HOME =", hadoop_home)
    else:
        print("WARNING: winutils.exe not found under", hadoop_home, "- local Parquet writes may fail")

os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
driver_memory = SPARK_CFG.get("driver_memory", "6g")
os.environ.setdefault(
    "PYSPARK_SUBMIT_ARGS",
    f"--driver-memory {driver_memory} --conf spark.driver.maxResultSize=2g pyspark-shell",
)

spark = (
    SparkSession.builder
    .master(SPARK_CFG.get("master", "local[*]"))
    .appName("hm-pyspark-analysis")
    .config("spark.sql.shuffle.partitions", str(SPARK_CFG.get("shuffle_partitions", 16)))
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "| master:", spark.sparkContext.master)


In [ ]:
'''
csv reader with schema
'''

def string_schema(fields: list[str]) -> StructType:
    return StructType([StructField(name, StringType(), True) for name in fields])


TRANSACTION_FIELDS = ["t_dat", "customer_id", "article_id", "price", "sales_channel_id"]
CUSTOMER_FIELDS = [
    "customer_id", "FN", "Active", "club_member_status",
    "fashion_news_frequency", "age", "postal_code",
]
ARTICLE_FIELDS = [
    "article_id", "product_code", "prod_name", "product_type_no", "product_type_name",
    "product_group_name", "graphical_appearance_no", "graphical_appearance_name",
    "colour_group_code", "colour_group_name", "perceived_colour_value_id",
    "perceived_colour_value_name", "perceived_colour_master_id",
    "perceived_colour_master_name", "department_no", "department_name", "index_code",
    "index_name", "index_group_no", "index_group_name", "section_no", "section_name",
    "garment_group_no", "garment_group_name", "detail_desc",
]


def read_csv(path: Path, fields: list[str], multi_line: bool = False, limit_rows: bool = False) -> DataFrame:
    reader = (
        spark.read
        .option("header", True)
        .option("mode", "PERMISSIVE")
        .option("multiLine", str(multi_line).lower())
        .schema(string_schema(fields))
    )
    df = reader.csv(str(path))
    return df.limit(ROW_LIMIT) if limit_rows and ROW_LIMIT > 0 else df


def clean_transactions(df: DataFrame) -> DataFrame:
    return (
        df.select(
            F.to_date("t_dat", "yyyy-MM-dd").alias("t_dat"),
            F.trim("customer_id").alias("customer_id"),
            F.trim("article_id").alias("article_id"),
            F.col("price").cast("decimal(12, 6)").alias("price"),
            F.col("sales_channel_id").cast("double").cast("int").alias("sales_channel_id"),
        )
        .where(
            F.col("t_dat").isNotNull()
            & F.col("customer_id").isNotNull()
            & F.col("article_id").isNotNull()
            & F.col("price").isNotNull()
            & (F.col("price") >= 0)
            & F.col("sales_channel_id").isin(1, 2)
        )
    )


def clean_customers(df: DataFrame) -> DataFrame:
    return (
        df.select(
            F.trim("customer_id").alias("customer_id"),
            F.col("FN").cast("double").cast("int").alias("FN"),
            F.col("Active").cast("double").cast("int").alias("Active"),
            F.trim("club_member_status").alias("club_member_status"),
            F.trim("fashion_news_frequency").alias("fashion_news_frequency"),
            F.col("age").cast("double").cast("int").alias("age"),
            F.trim("postal_code").alias("postal_code"),
        )
        .dropDuplicates(["customer_id"])
    )


ARTICLE_NUMERIC_FIELDS = {
    "product_type_no", "graphical_appearance_no", "perceived_colour_value_id",
    "perceived_colour_master_id", "department_no", "index_group_no",
    "section_no", "garment_group_no", "colour_group_code",
}


def clean_articles(df: DataFrame) -> DataFrame:
    selected = []
    for name in ARTICLE_FIELDS:
        if name == "article_id":
            selected.append(F.trim("article_id").alias("article_id"))
        elif name in ARTICLE_NUMERIC_FIELDS:
            selected.append(F.col(name).cast("double").cast("int").alias(name))
        else:
            selected.append(F.trim(name).alias(name))
    return df.select(*selected).dropDuplicates(["article_id"])


transactions = clean_transactions(read_csv(TRANSACTIONS_PATH, TRANSACTION_FIELDS, limit_rows=True)).cache()
customers = clean_customers(read_csv(CUSTOMERS_PATH, CUSTOMER_FIELDS)).cache()
articles = clean_articles(read_csv(ARTICLES_PATH, ARTICLE_FIELDS, multi_line=True)).cache()

print("loaded: transactions, customers, articles")


In [ ]:
print(f"transactions: {transactions.count():,}")
print(f"customers   : {customers.count():,}")
print(f"articles    : {articles.count():,}")

transactions.select(
    F.min("t_dat").alias("min_date"),
    F.max("t_dat").alias("max_date"),
    F.countDistinct("customer_id").alias("distinct_customers"),
    F.countDistinct("article_id").alias("distinct_articles"),
    F.sum("price").cast("double").alias("total_revenue"),
).show(truncate=False)




In [ ]:
'''
create detailed time column for analysis
'''
transactions = (transactions
.withColumn("year", F.year("t_dat"))
.withColumn("month", F.month("t_dat"))
.withColumn("quarter", F.quarter("t_dat"))
)